# OBPI Metric Debugging Notebook\n\nVisual validation of all 9 OBPI metrics on a synthetic match.\nRun this end-to-end after `pytest tests/` passes.

In [ ]:
import sys\nsys.path.insert(0, '../src')\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nfrom matplotlib.patches import Polygon as MplPolygon\n\nfrom obpi.data.loader import StatsBombLoader\nfrom obpi.pipeline import compute_all_metrics\nfrom obpi.utils.xt_model import XTModel\nfrom obpi.utils import geometry

## 1. Load or synthesise match data

In [ ]:
# Use synthetic data for offline reproducibility\nfrom tests.conftest import SyntheticMatchGenerator\n\ngen = SyntheticMatchGenerator(match_id=999999)\nevents = pd.DataFrame(gen.events(n_events=100))\nframes = gen.freeze_frames(n_frames=20)\n\nprint(f'Events: {len(events)} rows')\nprint(f'Frames: {len(frames)} frames')\nevents.head()

## 2. Run the full pipeline

In [ ]:
# Mock the loader so we don't hit the network\nfrom unittest.mock import MagicMock, patch\n\nwith patch('obpi.pipeline.StatsBombLoader') as MockLoader:\n    mock = MagicMock()\n    mock.get_events.return_value = events\n    mock.get_freeze_frames.return_value = frames\n    MockLoader.return_value = mock\n    df = compute_all_metrics(match_id=999999)\n\ndf

## 3. Metric Distributions

In [ ]:
metric_cols = [c for c in df.columns if c.startswith('M')]\ndf[metric_cols].describe().round(3)

## 4. Pitch Diagram: Voronoi Cells (M7 SCI)

In [ ]:
frame = frames[0]\nff = frame['freeze_frame']\nteammates = np.array([p['location'] for p in ff if p['teammate']], dtype=float)\nopponents = np.array([p['location'] for p in ff if not p['teammate']], dtype=float)\n\nfig, ax = plt.subplots(figsize=(12, 8))\nax.set_xlim(0, 120)\nax.set_ylim(0, 80)\nax.set_aspect('equal')\nax.set_title('Voronoi Cells — Synthetic Frame')\n\n# Plot points\nax.scatter(teammates[:, 0], teammates[:, 1], c='blue', label='Teammates', s=100)\nax.scatter(opponents[:, 0], opponents[:, 1], c='red', label='Opponents', s=100)\n\n# Draw pitch outline\nax.plot([0, 120, 120, 0, 0], [0, 0, 80, 80, 0], 'k-')\nax.legend()\nplt.show()

## 5. Receipt Heatmap: Half-space receipts (M5 RBTL)

In [ ]:
# Create a half-space polygon overlay\nhalf_space = geometry.get_half_space_polygon()\n\nfig, ax = plt.subplots(figsize=(12, 8))\nax.set_xlim(0, 120)\nax.set_ylim(0, 80)\nax.set_aspect('equal')\nax.set_title('Half-space Polygon (M5 RBTL)')\n\n# Draw half-space\nhs = MplPolygon(list(half_space.exterior.coords), alpha=0.2, color='green')\nax.add_patch(hs)\n\n# Plot receipt locations\nreceipts = events[events['type'].apply(lambda t: t.get('name') == 'Pass')]\nlocs = np.array(receipts['location'].tolist(), dtype=float)\nif len(locs):\n    ax.scatter(locs[:, 0], locs[:, 1], c='purple', alpha=0.5, label='Events')\n\nax.plot([0, 120, 120, 0, 0], [0, 0, 80, 80, 0], 'k-')\nax.legend()\nplt.show()

## 6. xT Grid Visualisation (M8 LPC)

In [ ]:
xt = XTModel()\ngrid = xt.get_grid()\n\nfig, ax = plt.subplots(figsize=(10, 6))\nim = ax.imshow(grid, origin='lower', cmap='YlOrRd', aspect='auto')\nax.set_title('Expected Threat (xT) 12x8 Grid')\nax.set_xlabel('Pitch x (10m cells)')\nax.set_ylabel('Pitch y (10m cells)')\nplt.colorbar(im, ax=ax, label='xT value')\nplt.show()

## 7. Metric Summary Table

In [ ]:
summary = df[['player_id'] + metric_cols].set_index('player_id')\nsummary.style.background_gradient(cmap='RdYlGn', axis=1).format('{:.2f}')